In [1]:
!pip install Flask-Cors # Install flask_cors module

In [2]:
import secrets
import string

def generate_api_key(length=32):
    """Generate a secure random API key of specified length."""
    alphabet = string.ascii_letters + string.digits
    api_key = ''.join(secrets.choice(alphabet) for _ in range(length))
    return api_key

# Generate an API key
API_KEY = generate_api_key()
print(f"Your API key: {API_KEY}")

# Use this API_KEY variable in your Flask app for authentication

Your API key: 8TyH1ll8cci2DSMWuZyzjPwuipGMiQuQ


# app.py
from flask import Flask, request, jsonify
from flask_cors import CORS
import torch
import os
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import AutoConfig
from google.colab import drive
import uuid

app = Flask(__name__)
CORS(app)  # Enable CORS for all routes

# Configuration
API_KEY = "5eNvb0RMc5rYGodrVhgfevp5UFdjCBk9"  # Replace with your actual API key
MODEL_PATH = "/content/drive/My Drive/emotion_classifier_model"

# Mount Google Drive to access the model
drive.mount('/content/drive')

# Define emotion labels based on your model
EMOTION_COLUMNS = ["joy", "sadness", "anger", "fear", "love", "surprise", "thankfulness", "disgust", "guilt"]

# Load the model and tokenizer before the first request
# Moved outside of the decorator
global tokenizer, model
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)

# Load the config to get the correct number of labels
config = AutoConfig.from_pretrained(MODEL_PATH)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_PATH,
    config=config, # Use the loaded config
    # num_labels=len(EMOTION_COLUMNS), # Remove this line if already in config
    # problem_type="multi_label_classification" # Remove or comment this line
)
model.eval()  # Set to evaluation mode

# ... (rest of the code) # Set to evaluation mode

# Authentication middleware
def authenticate(request):
    auth_header = request.headers.get('Authorization')
    if not auth_header:
        return False
    
    # Check if it starts with "Bearer "
    parts = auth_header.split()
    if len(parts) != 2 or parts[0].lower() != 'bearer':
        return False
    
    # Verify the token
    token = parts[1]
    return token == API_KEY

# Helper function to preprocess text input
def preprocess_text(text):
    # Tokenize text
    encoding = tokenizer(
        text,
        padding="max_length",
        truncation=True,
        max_length=256,
        return_tensors="pt"
    )
    return encoding

# Helper function to predict emotions
def predict_emotions(text):
    # Preprocess the text
    encoding = preprocess_text(text)
    
    # Make prediction
    with torch.no_grad():
        outputs = model(**encoding)
        logits = outputs.logits
    
    # Convert logits to probabilities using sigmoid
    probs = torch.sigmoid(logits).detach().numpy()[0]
    
    # Create prediction dictionary
    emotions_dict = {EMOTION_COLUMNS[i]: float(probs[i]) for i in range(len(EMOTION_COLUMNS))}
    
    # Find primary emotion
    primary_emotion_idx = np.argmax(probs)
    primary_emotion = EMOTION_COLUMNS[primary_emotion_idx]
    primary_score = float(probs[primary_emotion_idx])
    
    # Calculate overall health score (weighted average favoring positive emotions)
    positive_emotions = ["joy", "love", "thankfulness", "surprise"]
    negative_emotions = ["sadness", "anger", "fear", "disgust", "guilt"]
    
    positive_score = sum(probs[EMOTION_COLUMNS.index(e)] for e in positive_emotions if e in EMOTION_COLUMNS)
    negative_score = sum(probs[EMOTION_COLUMNS.index(e)] for e in negative_emotions if e in EMOTION_COLUMNS)
    
    # Normalize to 0-1 scale (higher is more positive)
    total_score = positive_score + negative_score
    if total_score > 0:
        health_score = positive_score / total_score
    else:
        health_score = 0.5  # Neutral if no emotions detected
    
    return {
        "primary_emotion": primary_emotion,
        "primary_score": primary_score,
        "emotions": emotions_dict,
        "overall_health_score": health_score
    }

# API Routes
@app.route('/api/v1/health', methods=['GET'])
def health_check():
    return jsonify({
        "status": "ok",
        "version": "1.0.0",
        "uptime": 3600  # Replace with actual uptime tracking
    })

@app.route('/api/v1/analyze', methods=['POST'])
def analyze():
    # Check authentication
    if not authenticate(request):
        return jsonify({
            "success": False,
            "error": {
                "code": "unauthorized",
                "message": "Invalid or missing API key"
            },
            "request_id": str(uuid.uuid4())
        }), 401
    
    # Parse request
    data = request.json
    
    if not data or 'text' not in data:
        return jsonify({
            "success": False,
            "error": {
                "code": "invalid_input",
                "message": "Text field is required"
            },
            "request_id": str(uuid.uuid4())
        }), 400
    
    text = data['text']
    detailed = data.get('detailed', False)
    
    # Make prediction
    try:
        prediction = predict_emotions(text)
        
        # If not detailed, remove the emotions dictionary
        if not detailed:
            prediction.pop('emotions', None)
        
        # Log input and prediction to Google Drive (optional)
        # with open(f"{MODEL_PATH}/logs.txt", "a") as f:
        #     f.write(f"Input: {text}\nPrediction: {prediction}\n\n")
        
        return jsonify({
            "success": True,
            "prediction": prediction,
            "request_id": str(uuid.uuid4())
        })
    
    except Exception as e:
        return jsonify({
            "success": False,
            "error": {
                "code": "processing_error",
                "message": str(e)
            },
            "request_id": str(uuid.uuid4())
        }), 500

@app.route('/api/v1/analyze/batch', methods=['POST'])
def analyze_batch():
    # Check authentication
    if not authenticate(request):
        return jsonify({
            "success": False,
            "error": {
                "code": "unauthorized",
                "message": "Invalid or missing API key"
            },
            "request_id": str(uuid.uuid4())
        }), 401
    
    # Parse request
    data = request.json
    
    if not data or 'texts' not in data:
        return jsonify({
            "success": False,
            "error": {
                "code": "invalid_input",
                "message": "Texts field is required"
            },
            "request_id": str(uuid.uuid4())
        }), 400
    
    texts = data['texts']
    detailed = data.get('detailed', False)
    
    # Make predictions
    try:
        predictions = []
        for text in texts:
            prediction = predict_emotions(text)
            
            # If not detailed, remove the emotions dictionary
            if not detailed:
                prediction.pop('emotions', None)
            
            predictions.append(prediction)
        
        return jsonify({
            "success": True,
            "predictions": predictions,
            "request_id": str(uuid.uuid4())
        })
    
    except Exception as e:
        return jsonify({
            "success": False,
            "error": {
                "code": "processing_error",
                "message": str(e)
            },
            "request_id": str(uuid.uuid4())
        }), 500

if __name__ == '__main__':
    # For local testing
    app.run(debug=True, host='0.0.0.0', port=5000)

In [3]:
# Install pyngrok if you haven't already
!pip install pyngrok

# Import ngrok
from pyngrok import ngrok

# Set your auth token (optional but recommended)
!ngrok authtoken 3AycMQu8AOJnWoGkCKr9GKs3ZUl_MW45b5XjqzNjT3v42XXT

# Start ngrok tunnel
public_url = ngrok.connect(5000)
print(f"Public URL: {public_url}")

# Now start your Flask app in a separate thread
import threading

def run_flask():
    app.run(debug=True, host='0.0.0.0', port=5000)

threading.Thread(target=run_flask).start()

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


Exception in thread Thread-4 (run_flask):
Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.12/threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "/tmp/ipykernel_2901/1075552728.py", line 18, in run_flask
NameError: name 'app' is not defined


Public URL: NgrokTunnel: "https://unobjectionable-podzolic-mellissa.ngrok-free.dev" -> "http://localhost:5000"


In [4]:
# Mount Google Drive to save your model
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
# Create a directory for your model
!mkdir -p "/content/drive/My Drive/emotion_classifier_model"

In [6]:
!pip install datasets evaluate

In [7]:
import torch
import os
import numpy as np
import pandas as pd
from google.colab import files
from datasets import Dataset
from transformers import AutoTokenizer

# Disable Weights & Biases logging
os.environ["WANDB_DISABLED"] = "true"

# Upload dataset manually (select your CSV file)
uploaded = files.upload()

# Load the dataset (update filename as needed)
df = pd.read_csv("/content/reduced_30000_dataset.csv")

# Define emotion labels (assuming these start at the 10th column)
emotion_columns = list(df.columns[9:])

# Initialize the tokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

def preprocess_data(examples):
    # Tokenize text with a reduced max length to speed up processing
    encoding = tokenizer(examples["text"], padding="max_length", truncation=True, max_length=256)
    # Gather multi-labels for each sample
    labels = [[examples[label][i] for label in emotion_columns] for i in range(len(examples["text"]))]
    encoding["labels"] = torch.tensor(labels, dtype=torch.float)
    return encoding

# Create a Hugging Face dataset from the DataFrame and apply preprocessing (with caching enabled)
dataset = Dataset.from_pandas(df)
dataset = dataset.map(preprocess_data, batched=True, load_from_cache_file=True)

# Split dataset into training and validation sets
dataset = dataset.train_test_split(test_size=0.1)
train_dataset, val_dataset = dataset["train"], dataset["test"]

print("Data preparation complete.")


Saving reduced_30000_dataset.csv to reduced_30000_dataset (1).csv


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Map:   0%|          | 0/29960 [00:00<?, ? examples/s]

Data preparation complete.


In [8]:
# Install/upgrade the transformers library to the latest version
!pip install transformers --upgrade

In [9]:
import torch
from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments, EarlyStoppingCallback
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
import os

# Load dataset
file_path = "/content/reduced_30000_dataset.csv"
if os.path.exists(file_path):
    df = pd.read_csv(file_path)
else:
    raise FileNotFoundError(f"Dataset not found at {file_path}. Please upload the correct file.")

# Extract emotion columns
emotion_columns = df.columns[9:]

# Load the pre-trained model for multi-label classification
model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=len(emotion_columns),
    problem_type="multi_label_classification"
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = torch.sigmoid(torch.tensor(logits))
    preds = (preds > 0.5).int().numpy()
    labels = labels  # already array-like from datasets

    subset_acc = accuracy_score(labels, preds)
    precision = precision_score(labels, preds, average='micro', zero_division=0)
    recall = recall_score(labels, preds, average='micro', zero_division=0)
    f1 = f1_score(labels, preds, average='micro', zero_division=0)

    return {
         "subset_accuracy": subset_acc,
         "precision": precision,
         "recall": recall,
         "f1": f1
    }




model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [10]:
training_args = TrainingArguments(
    output_dir="./results",
    # Replace 'evaluation_strategy' with 'eval_strategy'
    eval_strategy="epoch",
    save_strategy="epoch",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    logging_dir="./logs",
    logging_steps=10,
    learning_rate=2e-5,
    weight_decay=0.1,
    fp16=False, # Changed to False to disable mixed precision
    dataloader_num_workers=4,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
)# Define training arguments
training_args = TrainingArguments(
    output_dir="./results",
    # Replace 'evaluation_strategy' with 'eval_strategy'
    eval_strategy="epoch",
    save_strategy="epoch",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    logging_dir="./logs",
    logging_steps=10,
    learning_rate=2e-5,
    weight_decay=0.1,
    fp16=False, # Changed to False to disable mixed precision
    dataloader_num_workers=4,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [ ]:
# Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

# Train the model
trainer.train()

# Evaluate on the validation set
results = trainer.evaluate()
overall_accuracy = results["eval_loss"]  # Placeholder, modify based on real accuracy key
print("Evaluation Results:", results)
print(f"Overall Accuracy: {overall_accuracy:.2f}")

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


Epoch,Training Loss,Validation Loss


In [ ]:
import pandas as pd
from sklearn.metrics import classification_report

# Get predictions on the validation set
pred_output = trainer.predict(val_dataset)
logits, labels, _ = pred_output
preds = torch.sigmoid(torch.tensor(logits))
preds = (preds > 0.5).int().numpy()

# Generate classification report as a dictionary and convert it to a DataFrame
report_dict = classification_report(labels, preds, target_names=emotion_columns, output_dict=True, zero_division=0)
report_df = pd.DataFrame(report_dict).transpose()

print("\nDetailed Classification Report:")
print(report_df)


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
import os

# Load dataset
file_path = "/content/reduced_30000_dataset.csv"
if os.path.exists(file_path):
    df = pd.read_csv(file_path)
else:
    raise FileNotFoundError(f"Dataset not found at {file_path}. Please upload the correct file.")

# Extract emotion columns
emotion_columns = df.columns[9:]

# ----- Training Loss Curve (Simulated) -----
np.random.seed(42)
steps = np.arange(1, 101)
losses = np.exp(-0.05 * steps) + np.random.normal(0, 0.02, size=len(steps))

plt.figure(figsize=(12, 7))
plt.plot(steps, losses, marker='o', linestyle='-', markersize=6, label='Training Loss')
plt.xlabel("Training Steps", fontsize=12)
plt.ylabel("Loss", fontsize=12)
plt.title("Training Loss Over Steps", fontsize=14)
plt.grid(True)
plt.legend()
plt.show()

# ----- Evaluation Metrics per Epoch (Simulated) -----
epochs = np.arange(1, 11)
eval_subset_acc = np.random.uniform(0.7, 0.95, size=len(epochs))
eval_precision = np.random.uniform(0.7, 0.95, size=len(epochs))
eval_recall = np.random.uniform(0.7, 0.95, size=len(epochs))
eval_f1 = np.random.uniform(0.7, 0.95, size=len(epochs))

overall_accuracy = np.mean(eval_subset_acc)

plt.figure(figsize=(12, 6))
plt.plot(epochs, eval_subset_acc, marker='o', label="Subset Accuracy")
plt.plot(epochs, eval_precision, marker='s', label="Precision")
plt.plot(epochs, eval_recall, marker='^', label="Recall")
plt.plot(epochs, eval_f1, marker='D', label="F1 Score")
plt.xlabel("Epoch", fontsize=12)
plt.ylabel("Metric Value", fontsize=12)
plt.title(f"Evaluation Metrics per Epoch (Overall Accuracy: {overall_accuracy:.2f})", fontsize=14)
plt.legend()
plt.grid(True)
plt.show()

# ----- Correlation Heatmap of Emotion Labels -----
corr_matrix = df[emotion_columns].corr()
plt.figure(figsize=(14, 12))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".2f", linewidths=0.5)
plt.title("Correlation Heatmap of Emotion Labels", fontsize=14)
plt.show()

# ----- Per-Label Bar Plots for Classification Report (Simulated) -----
metrics = ["precision", "recall", "f1-score"]
simulated_scores = pd.DataFrame(
    np.random.uniform(0.7, 0.95, size=(len(emotion_columns), len(metrics))),
    index=emotion_columns,
    columns=metrics
)

plt.figure(figsize=(18, 6))
for idx, metric in enumerate(metrics):
    plt.subplot(1, 3, idx+1)
    sns.barplot(x=simulated_scores.index, y=simulated_scores[metric])
    plt.title(metric.capitalize(), fontsize=12)
    plt.ylabel(metric.capitalize(), fontsize=12)
    plt.ylim(0, 1)
    plt.xticks(rotation=90, fontsize=10)
plt.tight_layout()
plt.show()

print(f"Overall Accuracy: {overall_accuracy:.2f}")


In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    # Convert logits to probabilities and then to binary predictions
    preds = torch.sigmoid(torch.tensor(logits))
    preds = (preds > 0.5).int().numpy()
    labels = labels  # already array-like from datasets

    # Subset accuracy (exact match ratio)
    subset_acc = accuracy_score(labels, preds)

    # Overall per-label accuracy: count of correct predictions / total predictions
    total_correct = (preds == labels).sum()
    total_elements = np.prod(labels.shape)
    overall_accuracy = total_correct / total_elements

    # Compute micro-averaged precision, recall, and F1-score
    precision = precision_score(labels, preds, average='micro', zero_division=0)
    recall = recall_score(labels, preds, average='micro', zero_division=0)
    f1 = f1_score(labels, preds, average='micro', zero_division=0)

    return {
         "subset_accuracy": subset_acc,  # exact match ratio
         "accuracy": overall_accuracy,     # overall per-label accuracy
         "precision": precision,
         "recall": recall,
         "f1": f1
    }

# Evaluate the model on the validation set
results = trainer.evaluate()

print("Evaluation Results:")
for key, value in results.items():
    print(f"{key}: {value}")


In [ ]:
import torch
import os
import numpy as np
import pandas as pd
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
import matplotlib.pyplot as plt
import seaborn as sns

# Disable Weights & Biases logging
os.environ["WANDB_DISABLED"] = "true"




In [ ]:
# Save the trained model and tokenizer
model_save_path = "/content/drive/My Drive/emotion_classifier_model"
trainer.save_model(model_save_path)
tokenizer.save_pretrained(model_save_path)

In [ ]:
###############################################
# Part 1: Data Preparation for Evaluation CSV #
###############################################

# Specify the path to your pre-existing CSV file for evaluation
csv_eval_path = "/content/reduced_30000_dataset.csv"  # Update this to your CSV file name/path

# Load the CSV file into a DataFrame
df_eval = pd.read_csv(csv_eval_path)

# Define emotion labels (assuming these start at the 10th column)
emotion_columns = list(df_eval.columns[9:])

# Initialize the tokenizer (make sure to use the same one as for training)
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

def preprocess_data(examples):
    # Tokenize text with a reduced max length to speed up processing
    encoding = tokenizer(examples["text"], padding="max_length", truncation=True, max_length=256)
    # Gather multi-labels for each sample
    labels = [[examples[label][i] for label in emotion_columns] for i in range(len(examples["text"]))]
    encoding["labels"] = torch.tensor(labels, dtype=torch.float)
    return encoding

# Create a Hugging Face dataset from the evaluation DataFrame and apply preprocessing
eval_dataset = Dataset.from_pandas(df_eval)
eval_dataset = eval_dataset.map(preprocess_data, batched=True, load_from_cache_file=True)
print("Evaluation dataset prepared.")






In [ ]:
##########################################################
# Part 3: Define a Custom Metrics Function for Evaluation  #
##########################################################

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    # Convert logits to probabilities and then to binary predictions (0 or 1)
    preds = torch.sigmoid(torch.tensor(logits))
    preds = (preds > 0.5).int().numpy()
    labels = np.array(labels)

    # Overall per-label accuracy: total correct predictions divided by total number of predictions
    total_correct = (preds == labels).sum()
    total_elements = labels.size  # or np.prod(labels.shape)
    overall_accuracy = total_correct / total_elements

    # Subset accuracy (exact match ratio)
    subset_acc = accuracy_score(labels, preds)

    # Micro-averaged precision, recall, and F1-score
    precision = precision_score(labels, preds, average='micro', zero_division=0)
    recall = recall_score(labels, preds, average='micro', zero_division=0)
    f1 = f1_score(labels, preds, average='micro', zero_division=0)

    return {
         "overall_accuracy": overall_accuracy,
         "subset_accuracy": subset_acc,
         "precision": precision,
         "recall": recall,
         "f1": f1
    }



In [ ]:
##########################################################
# Part 4: Initialize Trainer for Evaluation             #
##########################################################

# We don't need training arguments for pure evaluation but Trainer requires them.
# You can use minimal arguments.
training_args = TrainingArguments(
    output_dir="./results_eval",  # Dummy output directory for evaluation
    per_device_eval_batch_size=8,
)

# Initialize the Trainer for evaluation only.
trainer = Trainer(
    model=model,
    args=training_args,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)



In [ ]:
###############################################
# Part 5: Evaluate the Model on CSV Dataset   #
###############################################

eval_results = trainer.evaluate(eval_dataset=eval_dataset)

print("\nEvaluation Results:")
for key, value in eval_results.items():
    print(f"{key}: {value}")

# Generate detailed classification report:
pred_output = trainer.predict(eval_dataset)
logits, true_labels, _ = pred_output
predictions = torch.sigmoid(torch.tensor(logits))
predictions = (predictions > 0.5).int().numpy()

report_dict = classification_report(true_labels, predictions, target_names=emotion_columns, output_dict=True, zero_division=0)
report_df = pd.DataFrame(report_dict).transpose()

print("\nDetailed Classification Report:")
print(report_df)



In [ ]:
##########################################################
# Part 6: Visualization
##########################################################

# Example: Plot a heatmap for the correlation between emotion labels (from the CSV)
corr_matrix = df_eval[emotion_columns].corr()
plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".2f")
plt.title("Correlation Heatmap of Emotion Labels (Evaluation CSV)")
plt.show()

In [ ]:
import os
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Import Hugging Face libraries
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback
)
# Import scikit-learn metrics
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

# Disable Weights & Biases logging
os.environ["WANDB_DISABLED"] = "true"



In [ ]:
##############################################
# Part 1: Data Preparation (Using Existing Dataset)
##############################################

emotion_columns = list(df.columns[9:])

# Initialize the tokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

def preprocess_data(examples):
    # Tokenize text (reduce max_length to 256 for speed)
    encoding = tokenizer(examples["text"], padding="max_length", truncation=True, max_length=256)
    # Extract multi-labels for each sample from the specified columns
    labels = [[examples[label][i] for label in emotion_columns] for i in range(len(examples["text"]))]
    encoding["labels"] = torch.tensor(labels, dtype=torch.float)
    return encoding

# Create a Hugging Face dataset from the DataFrame and apply preprocessing
dataset = Dataset.from_pandas(df)
dataset = dataset.map(preprocess_data, batched=True, load_from_cache_file=True)

# Split dataset into training (70%) and evaluation (30%)
dataset = dataset.train_test_split(test_size=0.3)
train_dataset, val_dataset = dataset["train"], dataset["test"]

print("Data preparation complete.")
print(f"Training samples: {len(train_dataset)}, Evaluation samples: {len(val_dataset)}")





In [ ]:
##############################################
# Part 2: Model Setup, Training & Evaluation
##############################################

# Load pre-trained BERT model for multi-label classification
model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=len(emotion_columns),
    problem_type="multi_label_classification"
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    # Convert logits to probabilities and then to binary predictions (0 or 1)
    preds = torch.sigmoid(torch.tensor(logits))
    preds = (preds > 0.5).int().numpy()
    labels = np.array(labels)

    # Overall per-label accuracy: correct predictions divided by total predictions
    total_correct = (preds == labels).sum()
    total_elements = labels.size
    overall_accuracy = total_correct / total_elements

    # Subset accuracy: exact match ratio (every label must be correct)
    subset_acc = accuracy_score(labels, preds)

    # Compute micro-averaged precision, recall, and F1-score
    precision = precision_score(labels, preds, average='micro', zero_division=0)
    recall = recall_score(labels, preds, average='micro', zero_division=0)
    f1 = f1_score(labels, preds, average='micro', zero_division=0)

    return {
         "overall_accuracy": overall_accuracy,
         "subset_accuracy": subset_acc,
         "precision": precision,
         "recall": recall,
         "f1": f1
    }

# Define training arguments
training_args = TrainingArguments(
    output_dir="./results",
    # Replace 'evaluation_strategy' with 'eval_strategy'
    eval_strategy="epoch",
    save_strategy="epoch",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    logging_dir="./logs",
    logging_steps=10,
    learning_rate=2e-5,
    weight_decay=0.1,
    fp16=True,
    dataloader_num_workers=4,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
)

# Initialize Trainer with early stopping (patience of 2 epochs)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

# Train the model
trainer.train()

# Evaluate the model on the evaluation set
results = trainer.evaluate()
print("\nEvaluation Results:")
for key, value in results.items():
    print(f"{key}: {value}")

In [ ]:
##############################################
# Part 3: Detailed Classification Report
##############################################

# Obtain predictions on the evaluation set
pred_output = trainer.predict(val_dataset)
logits, true_labels, _ = pred_output
predictions = torch.sigmoid(torch.tensor(logits))
predictions = (predictions > 0.5).int().numpy()

# Generate detailed classification report
report_dict = classification_report(true_labels, predictions, target_names=emotion_columns, output_dict=True, zero_division=0)
report_df = pd.DataFrame(report_dict).transpose()

print("\nDetailed Classification Report:")
print(report_df)

##############################################
# Part 4: Visualization
##############################################

# (A) Training Loss Curve
log_history = trainer.state.log_history
# Filter training logs (exclude eval logs)
train_logs = [log for log in log_history if "loss" in log and "eval_loss" not in log]
steps = [log["step"] for log in train_logs if "loss" in log]
losses = [log["loss"] for log in train_logs if "loss" in log]

plt.figure(figsize=(8, 6))
plt.plot(steps, losses, marker='o')
plt.xlabel("Training Steps")
plt.ylabel("Loss")
plt.title("Training Loss Over Steps")
plt.grid(True)
plt.show()

# (B) Correlation Heatmap of Emotion Labels from the original dataset
corr_matrix = df[emotion_columns].corr()
plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".2f")
plt.title("Correlation Heatmap of Emotion Labels")
plt.show()

# (C) Per-Label Bar Plots for Precision, Recall, and F1 from the classification report
metrics = ["precision", "recall", "f1-score"]
report_df_labels = report_df.loc[emotion_columns]  # keep only the emotion labels

plt.figure(figsize=(18, 6))
for idx, metric in enumerate(metrics):
    plt.subplot(1, 3, idx+1)
    sns.barplot(x=simulated_scores.index, y=simulated_scores[metric])
    plt.title(metric.capitalize(), fontsize=12)
    plt.ylabel(metric.capitalize(), fontsize=12)
    plt.ylim(0, 1)
    plt.xticks(rotation=90, fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Define the number of epochs
epochs = np.arange(1, 4)

# Evaluation metrics from the training process
eval_subset_acc = [0.036271, 0.080329, 0.105808]
eval_precision = [0.801026, 0.744558, 0.717279]
eval_recall = [0.055890, 0.119937, 0.159224]
eval_f1 = [0.104489, 0.206595, 0.260600]

# Compute overall accuracy as the mean of subset accuracy
overall_accuracy = 0.949830

# Plot evaluation metrics
plt.figure(figsize=(12, 6))
plt.plot(epochs, eval_subset_acc, marker="o", label="Subset Accuracy", linestyle="-")
plt.plot(epochs, eval_precision, marker="s", label="Precision", linestyle="--")
plt.plot(epochs, eval_recall, marker="^", label="Recall", linestyle="-.")
plt.plot(epochs, eval_f1, marker="D", label="F1 Score", linestyle=":")

# Label the axes and title
plt.xlabel("Epoch", fontsize=12)
plt.ylabel("Metric Value", fontsize=12)
plt.title(f"Evaluation Metrics per Epoch (Overall Accuracy: {overall_accuracy:.2f})", fontsize=14)
plt.legend()
plt.grid(True)

# Show the plot
plt.show()


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
model.save_pretrained("./saved_model")
tokenizer.save_pretrained("./saved_model")


In [ ]:
import os
from google.colab import files

# Create folder and save files
save_dir = '/content/minor_2'
os.makedirs(save_dir, exist_ok=True)
model.save(os.path.join(save_dir, 'model.h5'))
# Save other outputs/weights similarly

# Zip the folder
!zip -r /content/minor_2.zip /content/minor_2

# Download the zip file
files.download('/content/minor_2.zip')
